# GCH23 Rurality and Remoteness Experiment

## Objective

This experiment investigates whether rurality and remoteness provide
additional predictive information for classifying the severity of recorded
New Zealand road crashes.

The prediction target remains `crashSeverity`:

- Non-Injury Crash
- Minor Crash
- Serious Crash
- Fatal Crash

The Geographic Classification for Health 2023 (GCH23) provides a more
detailed representation of rurality than the existing binary CAS `urban`
variable.

GCH23 contains five categories:

- U1 — Urban 1
- U2 — Urban 2
- R1 — Rural 1
- R2 — Rural 2
- R3 — Rural 3

The project does not predict crash occurrence or crash risk.

In [1]:
# Load and inspect manually downloaded GCH23 workbook

import pandas as pd
from pathlib import Path


# 1. Find project root automatically

CURRENT_FOLDER = Path.cwd()

PROJECT_ROOT = None

for folder in [CURRENT_FOLDER, *CURRENT_FOLDER.parents]:

    candidate = (
        folder
        / "data"
        / "gch23"
        / "MoH-MB-to-GCH2023.xlsx"
    )

    if candidate.exists():
        PROJECT_ROOT = folder
        break


if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root containing "
        "data/gch23/MoH-MB-to-GCH2023.xlsx."
    )


GCH_PATH = (
    PROJECT_ROOT
    / "data"
    / "gch23"
    / "MoH-MB-to-GCH2023.xlsx"
)


# 2. Confirm file exists

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "File exists:",
    GCH_PATH.exists()
)

print(
    "File path:"
)

print(
    GCH_PATH
)


if not GCH_PATH.exists():
    raise FileNotFoundError(
        f"GCH23 file not found:\n{GCH_PATH}"
    )


print(
    "\nFile size:",
    round(
        GCH_PATH.stat().st_size
        / 1024
        / 1024,
        2
    ),
    "MB"
)


# 3. Inspect workbook

excel_file = pd.ExcelFile(
    GCH_PATH
)

print(
    "\nSheet names:"
)

print(
    excel_file.sheet_names
)


# 4. Inspect each sheet

for sheet in excel_file.sheet_names:

    print(
        "\nSheet:",
        sheet
    )

    preview = pd.read_excel(
        GCH_PATH,
        sheet_name=sheet,
        nrows=10
    )

    print(
        "\nColumns:"
    )

    print(
        preview.columns.tolist()
    )

    print(
        "\nFirst rows:"
    )

    display(
        preview.head(10)
    )

Project root: C:\Users\ASUS\Desktop\Project_Roadcrash
File exists: True
File path:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\gch23\MoH-MB-to-GCH2023.xlsx

File size: 1.1 MB

Sheet names:
['MoH_MB_to_GCH2023', 'Sheet1']

Sheet: MoH_MB_to_GCH2023

Columns:
['MB_code', 'MB_version', 'GCH2023']

First rows:


,MB_code,MB_version,GCH2023
0,100,2023,R3
1,200,2023,R3
2,300,2023,R3
3,400,2023,R3
4,501,2023,Inlet
5,502,2023,R3
6,600,2023,R3
7,700,2023,R3
8,800,2023,R3
9,900,2023,R3



Sheet: Sheet1

Columns:
[]

First rows:


""


In [2]:
# Load full GCH23 concordance and join to crashes
# using previously validated spatial MB2023 assignment

import pandas as pd
from pathlib import Path


# 1. Find project root automatically

CURRENT_FOLDER = Path.cwd()

PROJECT_ROOT = None

for folder in [CURRENT_FOLDER, *CURRENT_FOLDER.parents]:

    gch_candidate = (
        folder
        / "data"
        / "gch23"
        / "MoH-MB-to-GCH2023.xlsx"
    )

    spatial_candidate = (
        folder
        / "data"
        / "nzdep2023"
        / "CAS_NZDep2023_enriched.csv"
    )

    if (
        gch_candidate.exists()
        and spatial_candidate.exists()
    ):
        PROJECT_ROOT = folder
        break


if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root containing both "
        "data/gch23/MoH-MB-to-GCH2023.xlsx and "
        "data/nzdep2023/CAS_NZDep2023_enriched.csv."
    )


GCH_PATH = (
    PROJECT_ROOT
    / "data"
    / "gch23"
    / "MoH-MB-to-GCH2023.xlsx"
)

SPATIAL_MB_PATH = (
    PROJECT_ROOT
    / "data"
    / "nzdep2023"
    / "CAS_NZDep2023_enriched.csv"
)


print(
    "Project root:",
    PROJECT_ROOT
)


# 2. Load full GCH23 sheet

gch = pd.read_excel(
    GCH_PATH,
    sheet_name="MoH_MB_to_GCH2023"
)


print(
    "GCH23 shape:"
)

print(
    gch.shape
)

print(
    "\nColumns:"
)

print(
    gch.columns.tolist()
)


# 3. Validate geography version

print(
    "\nMB_version distribution:"
)

print(
    gch["MB_version"]
    .value_counts(
        dropna=False
    )
    .sort_index()
)


# 4. Normalise meshblock key

gch[
    "meshblock_2023_key"
] = (
    pd.to_numeric(
        gch[
            "MB_code"
        ],
        errors="coerce"
    )
    .astype("Int64")
)


print(
    "\nUnique meshblock keys:"
)

print(
    gch[
        "meshblock_2023_key"
    ]
    .nunique()
)

print(
    "\nDuplicate meshblock keys:"
)

print(
    gch[
        "meshblock_2023_key"
    ]
    .duplicated()
    .sum()
)

print(
    "\nMissing meshblock keys:"
)

print(
    gch[
        "meshblock_2023_key"
    ]
    .isna()
    .sum()
)


# 5. Inspect all GCH23 categories

print(
    "\nFull GCH2023 Category Distribution"
)

print(
    gch[
        "GCH2023"
    ]
    .value_counts(
        dropna=False
    )
)


# 6. Load previously spatially assigned crash MB2023

crash_mb_saved = pd.read_csv(
    SPATIAL_MB_PATH,
    low_memory=False
)


crash_mb_saved[
    "meshblock_2023_key"
] = (
    pd.to_numeric(
        crash_mb_saved[
            "MB2023_V1_00"
        ],
        errors="coerce"
    )
    .astype("Int64")
)


print(
    "\nSaved crash spatial dataset shape:"
)

print(
    crash_mb_saved.shape
)

print(
    "Unique crash IDs:",
    crash_mb_saved[
        "cas_row_id"
    ].nunique()
)


# 7. Join GCH23 to crash records

crash_gch = crash_mb_saved.merge(
    gch[
        [
            "meshblock_2023_key",
            "GCH2023"
        ]
    ],
    on="meshblock_2023_key",
    how="left",
    validate="many_to_one"
)


# 8. Validate merge

print(
    "\nGCH23 Crash Merge Validation"
)

print(
    "Rows before merge:",
    len(crash_mb_saved)
)

print(
    "Rows after merge:",
    len(crash_gch)
)

print(
    "Duplicate cas_row_id:",
    crash_gch[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)


# 9. Coverage

gch_available = (
    crash_gch[
        "GCH2023"
    ]
    .notna()
)

print(
    "\nCrashes with GCH2023:",
    gch_available.sum()
)

print(
    "Crashes without GCH2023:",
    (~gch_available).sum()
)

print(
    "GCH2023 coverage:",
    round(
        gch_available.mean() * 100,
        2
    ),
    "%"
)


# 10. Crash-level GCH distribution

print(
    "\nCrash-Level GCH2023 Distribution"
)

print(
    crash_gch[
        "GCH2023"
    ]
    .value_counts(
        dropna=False
    )
)

Project root: C:\Users\ASUS\Desktop\Project_Roadcrash
GCH23 shape:
(67604, 3)

Columns:
['MB_code', 'MB_version', 'GCH2023']

MB_version distribution:
MB_version
2001     1565
2006     1908
2013     4744
2018     1849
2023    57538
Name: count, dtype: int64

Unique meshblock keys:
67604

Duplicate meshblock keys:
0

Missing meshblock keys:
0

Full GCH2023 Category Distribution
GCH2023
U1         37414
U2         11733
R1         10249
R2          5693
R3          1739
Inlet        529
Oceanic      247
Name: count, dtype: int64

Saved crash spatial dataset shape:
(34866, 9)
Unique crash IDs: 34866

GCH23 Crash Merge Validation
Rows before merge: 34866
Rows after merge: 34866
Duplicate cas_row_id: 0

Crashes with GCH2023: 34866
Crashes without GCH2023: 0
GCH2023 coverage: 100.0 %

Crash-Level GCH2023 Distribution
GCH2023
U1         20044
U2          6413
R1          5029
R2          2505
R3           754
Inlet        102
Oceanic       19
Name: count, dtype: int64


In [3]:
# Rebuild clean GCH23 crash dataset
# Use only 2023 GCH meshblocks

import pandas as pd
import numpy as np


# 1. Keep only GCH meshblocks from 2023

gch_2023 = (
    gch[
        gch["MB_version"] == 2023
    ]
    .copy()
)

gch_2023[
    "meshblock_2023_key"
] = (
    pd.to_numeric(
        gch_2023[
            "MB_code"
        ],
        errors="coerce"
    )
    .astype("Int64")
)


print(
    "GCH 2023 rows:",
    len(gch_2023)
)

print(
    "Unique 2023 meshblock keys:",
    gch_2023[
        "meshblock_2023_key"
    ].nunique()
)

print(
    "Duplicate 2023 meshblock keys:",
    gch_2023[
        "meshblock_2023_key"
    ]
    .duplicated()
    .sum()
)


# 2. Reload spatial crash dataset
# This file already contains crashSeverity.

crash_mb_saved = pd.read_csv(
    SPATIAL_MB_PATH,
    low_memory=False
)

crash_mb_saved[
    "meshblock_2023_key"
] = (
    pd.to_numeric(
        crash_mb_saved[
            "MB2023_V1_00"
        ],
        errors="coerce"
    )
    .astype("Int64")
)


# 3. Join only GCH2023

crash_gch = crash_mb_saved.merge(
    gch_2023[
        [
            "meshblock_2023_key",
            "GCH2023"
        ]
    ],
    on="meshblock_2023_key",
    how="left",
    validate="many_to_one"
)


# 4. Add only the CAS urban variable
# Do not merge crashSeverity again.

CAS_PATH = (
    PROJECT_ROOT
    / "data"
    / "CAS_Data_public.csv"
)

cas = pd.read_csv(
    CAS_PATH,
    low_memory=False
)

cas[
    "cas_row_id"
] = np.arange(
    len(cas)
)


crash_gch = crash_gch.merge(
    cas[
        [
            "cas_row_id",
            "urban"
        ]
    ],
    on="cas_row_id",
    how="left",
    validate="one_to_one"
)


# 5. Validate required columns

print(
    "\nRequired columns present:"
)

for col in [
    "crashSeverity",
    "urban",
    "GCH2023"
]:
    print(
        col,
        "->",
        col in crash_gch.columns
    )


print(
    "\nRows:",
    len(crash_gch)
)

print(
    "Duplicate crash IDs:",
    crash_gch[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)

print(
    "Missing GCH2023:",
    crash_gch[
        "GCH2023"
    ]
    .isna()
    .sum()
)


# 6. Clean GCH classes

gch_order = [
    "U1",
    "U2",
    "R1",
    "R2",
    "R3"
]


crash_gch[
    "GCH2023_clean"
] = (
    crash_gch[
        "GCH2023"
    ]
    .where(
        crash_gch[
            "GCH2023"
        ]
        .isin(
            gch_order
        ),
        "Other"
    )
)


# 7. Ordinal representation

gch_ordinal_map = {
    "U1": 1,
    "U2": 2,
    "R1": 3,
    "R2": 4,
    "R3": 5
}

crash_gch[
    "GCH_rurality_ordinal"
] = (
    crash_gch[
        "GCH2023"
    ]
    .map(
        gch_ordinal_map
    )
)


crash_gch[
    "GCH_special_geography"
] = (
    ~crash_gch[
        "GCH2023"
    ]
    .isin(
        gch_order
    )
).astype(int)


# 8. Clean GCH distribution

print(
    "\nClean GCH23 Distribution"
)

print(
    crash_gch[
        "GCH2023_clean"
    ]
    .value_counts(
        dropna=False
    )
)


# 9. Compare GCH with existing CAS urban

print(
    "\nGCH23 vs CAS urban"
)

gch_vs_urban = pd.crosstab(
    crash_gch[
        "GCH2023_clean"
    ],
    crash_gch[
        "urban"
    ],
    margins=True
)

display(
    gch_vs_urban
)


# 10. Severity counts by GCH

print(
    "\nCrash Severity by GCH23"
)

severity_gch = pd.crosstab(
    crash_gch[
        "GCH2023_clean"
    ],
    crash_gch[
        "crashSeverity"
    ]
)

display(
    severity_gch
)


# 11. Severity percentages within each GCH group

print(
    "\nSeverity Percentage Within Each GCH Group"
)

severity_gch_pct = (
    pd.crosstab(
        crash_gch[
            "GCH2023_clean"
        ],
        crash_gch[
            "crashSeverity"
        ],
        normalize="index"
    )
    * 100
).round(2)

display(
    severity_gch_pct
)


# 12. Ordinal rurality by crash severity

print(
    "\nGCH Ordinal Rurality by Crash Severity"
)

gch_severity_summary = (
    crash_gch[
        crash_gch[
            "GCH_rurality_ordinal"
        ].notna()
    ]
    .groupby(
        "crashSeverity"
    )
    .agg(
        Count=(
            "GCH_rurality_ordinal",
            "count"
        ),
        Rurality_Mean=(
            "GCH_rurality_ordinal",
            "mean"
        ),
        Rurality_Median=(
            "GCH_rurality_ordinal",
            "median"
        )
    )
    .round(3)
)

display(
    gch_severity_summary
)

GCH 2023 rows: 57538
Unique 2023 meshblock keys: 57538
Duplicate 2023 meshblock keys: 0

Required columns present:
crashSeverity -> True
urban -> True
GCH2023 -> True

Rows: 34866
Duplicate crash IDs: 0
Missing GCH2023: 0

Clean GCH23 Distribution
GCH2023_clean
U1       20044
U2        6413
R1        5029
R2        2505
R3         754
Other      121
Name: count, dtype: int64

GCH23 vs CAS urban


urban,Open,Urban,All
GCH2023_clean,,,
Other,88,33,121
R1,3201,1828,5029
R2,1764,741,2505
R3,659,95,754
U1,4861,15183,20044
U2,1888,4525,6413
All,12461,22405,34866



Crash Severity by GCH23


crashSeverity,Fatal Crash,Minor Crash,Non-Injury Crash,Serious Crash
GCH2023_clean,,,,
Other,3,36,63,19
R1,86,1526,2934,483
R2,54,788,1396,267
R3,12,214,426,102
U1,99,5671,13110,1164
U2,73,1762,4107,471



Severity Percentage Within Each GCH Group


crashSeverity,Fatal Crash,Minor Crash,Non-Injury Crash,Serious Crash
GCH2023_clean,,,,
Other,2.48,29.75,52.07,15.70
R1,1.71,30.34,58.34,9.60
R2,2.16,31.46,55.73,10.66
R3,1.59,28.38,56.50,13.53
U1,0.49,28.29,65.41,5.81
U2,1.14,27.48,64.04,7.34



GCH Ordinal Rurality by Crash Severity


,Count,Rurality_Mean,Rurality_Median
crashSeverity,,,
Fatal Crash,324,2.404,2.0
Minor Crash,9961,1.807,1.0
Non-Injury Crash,21973,1.722,1.0
Serious Crash,2487,2.064,2.0


### GCH23 Exploratory Findings

The GCH23 classification showed a clearer descriptive relationship with
crash severity than the previously evaluated socioeconomic deprivation
measure.

The proportion of Serious and Fatal crashes generally increased across
progressively more rural GCH categories. Serious-or-Fatal crashes represented
approximately 6.3% of U1 crashes, 8.5% of U2 crashes, 11.3% of R1 crashes,
12.8% of R2 crashes, and 15.1% of R3 crashes.

The mean ordinal rurality value was also higher for Fatal crashes (2.404)
and Serious crashes (2.064) than for Minor (1.807) and Non-Injury crashes
(1.722).

These results describe associations within recorded crash data and do not
establish that rurality causes greater crash severity.

In [4]:
# Statistical association between GCH23 and crash severity

import numpy as np
import pandas as pd

from scipy.stats import (
    chi2_contingency,
    kruskal
)


# 1. Use only the five official ordered GCH classes

gch_valid = crash_gch[
    crash_gch["GCH2023"].isin(
        ["U1", "U2", "R1", "R2", "R3"]
    )
].copy()


print(
    "Crashes in official U1-U2-R1-R2-R3 classes:",
    len(gch_valid)
)

print(
    "Excluded special-geography crashes:",
    len(crash_gch) - len(gch_valid)
)


# 2. Chi-square association
# GCH category × crash severity

contingency = pd.crosstab(
    gch_valid["GCH2023"],
    gch_valid["crashSeverity"]
)

chi2, p_chi, dof, expected = (
    chi2_contingency(
        contingency
    )
)


# 3. Cramer's V effect size

n = contingency.to_numpy().sum()

r, k = contingency.shape

cramers_v = np.sqrt(
    chi2
    /
    (
        n * min(r - 1, k - 1)
    )
)


print(
    "\nChi-Square Association"
)

print(
    "Chi-square:",
    round(
        chi2,
        4
    )
)

print(
    "Degrees of freedom:",
    dof
)

print(
    "p-value:",
    p_chi
)

print(
    "Cramer's V:",
    round(
        cramers_v,
        4
    )
)


# 4. Kruskal-Wallis test on ordinal rurality

groups = [
    group[
        "GCH_rurality_ordinal"
    ]
    .dropna()
    .values

    for _, group
    in gch_valid.groupby(
        "crashSeverity"
    )
]


H, p_kw = kruskal(
    *groups
)

n_kw = sum(
    len(group)
    for group in groups
)

k_kw = len(
    groups
)

epsilon_squared = max(
    0,
    (
        H - k_kw + 1
    )
    /
    (
        n_kw - k_kw
    )
)


print(
    "\nKruskal-Wallis: Rurality by Severity"
)

print(
    "H statistic:",
    round(
        H,
        4
    )
)

print(
    "p-value:",
    p_kw
)

print(
    "Epsilon-squared:",
    round(
        epsilon_squared,
        4
    )
)


# 5. Combined severe-crash percentage

severity_pct = (
    pd.crosstab(
        gch_valid[
            "GCH2023"
        ],
        gch_valid[
            "crashSeverity"
        ],
        normalize="index"
    )
    * 100
)


severity_pct[
    "Fatal + Serious %"
] = (
    severity_pct[
        "Fatal Crash"
    ]
    +
    severity_pct[
        "Serious Crash"
    ]
)


severity_pct = (
    severity_pct.loc[
        [
            "U1",
            "U2",
            "R1",
            "R2",
            "R3"
        ]
    ]
)


print(
    "\nSevere Crash Percentage by GCH23"
)

display(
    severity_pct[
        [
            "Fatal Crash",
            "Serious Crash",
            "Fatal + Serious %"
        ]
    ].round(2)
)

Crashes in official U1-U2-R1-R2-R3 classes: 34745
Excluded special-geography crashes: 121

Chi-Square Association
Chi-square: 377.9333
Degrees of freedom: 12
p-value: 1.7667000808302477e-73
Cramer's V: 0.0602

Kruskal-Wallis: Rurality by Severity
H statistic: 332.0171
p-value: 1.1674272252598968e-71
Epsilon-squared: 0.0095

Severe Crash Percentage by GCH23


crashSeverity,Fatal Crash,Serious Crash,Fatal + Serious %
GCH2023,,,
U1,0.49,5.81,6.30
U2,1.14,7.34,8.48
R1,1.71,9.60,11.31
R2,2.16,10.66,12.81
R3,1.59,13.53,15.12


### Statistical Association Between GCH23 and Crash Severity

GCH23 showed a statistically significant association with recorded crash
severity (chi-square = 377.93, p < 0.001). The corresponding Cramer's V
was 0.0602, indicating that the overall categorical association was small.

The ordered rurality representation also differed significantly across
severity classes (Kruskal-Wallis H = 332.02, p < 0.001), with an
epsilon-squared effect size of 0.0095.

Although the effect remains small, it was substantially larger than the
univariate effect observed for NZDep2023 deprivation decile.

A particularly clear descriptive pattern was observed for severe outcomes.
The combined proportion of Fatal and Serious crashes increased from 6.30%
in U1 to 8.48% in U2, 11.31% in R1, 12.81% in R2, and 15.12% in R3.

These findings indicate that rurality provides potentially useful contextual
information for crash-severity classification. However, the relationships
are observational and should not be interpreted as causal effects of
rurality on crash severity.

In [5]:
# GCH23 Feature Ablation
# Weighted XGBoost: Seed 42
# Full national CAS population

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

from xgboost import XGBClassifier


# 1. Original 30 CAS baseline predictors

baseline_features = [
    "bicycle",
    "bus",
    "carStationWagon",
    "crashDirectionDescription",
    "crashFinancialYear",
    "crashSHDescription",
    "crashYear",
    "directionRoleDescription",
    "flatHill",
    "light",
    "moped",
    "motorcycle",
    "NumberOfLanes",
    "otherVehicleType",
    "region",
    "roadCharacter",
    "roadLane",
    "roadSurface",
    "schoolBus",
    "speedLimit",
    "streetLight",
    "suv",
    "taxi",
    "trafficControl",
    "truck",
    "unknownVehicleType",
    "urban",
    "vanOrUtility",
    "weatherA",
    "weatherB"
]

target = "crashSeverity"


baseline_categorical = [
    "crashDirectionDescription",
    "crashFinancialYear",
    "crashSHDescription",
    "directionRoleDescription",
    "flatHill",
    "light",
    "region",
    "roadCharacter",
    "roadLane",
    "roadSurface",
    "streetLight",
    "trafficControl",
    "urban",
    "weatherA",
    "weatherB"
]


# 2. Rebuild modelling dataframe
# crash_gch contains GCH variables but not all original CAS predictors,
# so merge GCH features back to original CAS using cas_row_id.

gch_features_for_merge = crash_gch[
    [
        "cas_row_id",
        "GCH2023_clean",
        "GCH_rurality_ordinal",
        "GCH_special_geography"
    ]
].copy()


gch_model_df = cas.merge(
    gch_features_for_merge,
    on="cas_row_id",
    how="left",
    validate="one_to_one"
)


print(
    "GCH modelling dataframe:",
    gch_model_df.shape
)

print(
    "Missing GCH categorical:",
    gch_model_df[
        "GCH2023_clean"
    ]
    .isna()
    .sum()
)

print(
    "Missing GCH ordinal:",
    gch_model_df[
        "GCH_rurality_ordinal"
    ]
    .isna()
    .sum()
)


# 3. Feature configurations

baseline_without_urban = [
    col
    for col in baseline_features
    if col != "urban"
]


gch_feature_sets = {

    "CAS Only":
        baseline_features,

    "CAS + GCH Categorical":
        baseline_features
        + ["GCH2023_clean"],

    "CAS + GCH Ordinal":
        baseline_features
        + ["GCH_rurality_ordinal"],

    "CAS + GCH Ordinal + Special Indicator":
        baseline_features
        + [
            "GCH_rurality_ordinal",
            "GCH_special_geography"
        ],

    "CAS Without Urban + GCH Categorical":
        baseline_without_urban
        + ["GCH2023_clean"]
}


# 4. Fixed seed-42 train/test split

all_indices = gch_model_df.index

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=42,
    stratify=gch_model_df.loc[
        all_indices,
        target
    ]
)


y_train = gch_model_df.loc[
    train_idx,
    target
]

y_test = gch_model_df.loc[
    test_idx,
    target
]


encoder = LabelEncoder()

y_train_encoded = (
    encoder.fit_transform(
        y_train
    )
)

y_test_encoded = (
    encoder.transform(
        y_test
    )
)


fatal_id = list(
    encoder.classes_
).index(
    "Fatal Crash"
)

serious_id = list(
    encoder.classes_
).index(
    "Serious Crash"
)


sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_encoded
)


print(
    "\nTraining crashes:",
    len(train_idx)
)

print(
    "Test crashes:",
    len(test_idx)
)

print(
    "\nClass mapping:"
)

for i, label in enumerate(
    encoder.classes_
):
    print(
        i,
        "=",
        label
    )


# 5. Evaluation function

def evaluate_gch_features(
    experiment_name,
    feature_list
):

    X_train = gch_model_df.loc[
        train_idx,
        feature_list
    ].copy()

    X_test = gch_model_df.loc[
        test_idx,
        feature_list
    ].copy()


    # GCH categorical must be treated as categorical

    categorical_list = [
        col
        for col in feature_list
        if (
            col in baseline_categorical
            or col == "GCH2023_clean"
        )
    ]


    numeric_list = [
        col
        for col in feature_list
        if col not in categorical_list
    ]


    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])


    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ])


    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            numeric_list
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_list
        )
    ])


    X_train_processed = (
        preprocessor.fit_transform(
            X_train
        )
    )

    X_test_processed = (
        preprocessor.transform(
            X_test
        )
    )


    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )


    model.fit(
        X_train_processed,
        y_train_encoded,
        sample_weight=sample_weights
    )


    predictions = model.predict(
        X_test_processed
    )


    fatal_recall = recall_score(
        y_test_encoded,
        predictions,
        labels=[fatal_id],
        average=None,
        zero_division=0
    )[0]


    serious_recall = recall_score(
        y_test_encoded,
        predictions,
        labels=[serious_id],
        average=None,
        zero_division=0
    )[0]


    return {

        "Feature Set":
            experiment_name,

        "Predictors":
            len(feature_list),

        "Accuracy":
            accuracy_score(
                y_test_encoded,
                predictions
            ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_test_encoded,
                predictions
            ),

        "Macro F1":
            f1_score(
                y_test_encoded,
                predictions,
                average="macro"
            ),

        "Fatal Recall":
            fatal_recall,

        "Serious Recall":
            serious_recall
    }


# 6. Run experiments

gch_results = []

for experiment_name, features in (
    gch_feature_sets.items()
):

    print(
        "\nRunning:",
        experiment_name
    )

    result = evaluate_gch_features(
        experiment_name,
        features
    )

    gch_results.append(
        result
    )


# 7. Results

gch_ablation = pd.DataFrame(
    gch_results
)


baseline_macro = gch_ablation.loc[
    gch_ablation[
        "Feature Set"
    ] == "CAS Only",
    "Macro F1"
].iloc[0]


gch_ablation[
    "Macro F1 Change"
] = (
    gch_ablation[
        "Macro F1"
    ]
    - baseline_macro
)


gch_ablation = (
    gch_ablation
    .sort_values(
        "Macro F1",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    gch_ablation.round(4)
)

GCH modelling dataframe: (34866, 76)
Missing GCH categorical: 0
Missing GCH ordinal: 121

Training crashes: 27892
Test crashes: 6974

Class mapping:
0 = Fatal Crash
1 = Minor Crash
2 = Non-Injury Crash
3 = Serious Crash

Running: CAS Only

Running: CAS + GCH Categorical

Running: CAS + GCH Ordinal

Running: CAS + GCH Ordinal + Special Indicator

Running: CAS Without Urban + GCH Categorical


,Feature Set,Predictors,Accuracy,Balanced Accuracy,Macro F1,Fatal Recall,Serious Recall,Macro F1 Change
0,CAS + GCH Ordinal,31,0.4984,0.3948,0.3303,0.2462,0.3892,0.0030
1,CAS + GCH Categorical,31,0.4983,0.3922,0.3299,0.2308,0.3992,0.0027
2,CAS + GCH Ordinal + Special Indicator,32,0.4947,0.3991,0.3296,0.2615,0.4052,0.0023
3,CAS Without Urban + GCH Categorical,30,0.4953,0.3978,0.3290,0.2615,0.3972,0.0018
4,CAS Only,30,0.4900,0.4076,0.3272,0.3231,0.3892,0.0000


### GCH23 Feature-Ablation Result

At random seed 42, GCH23 produced modest improvements in Macro F1 over
the CAS-only baseline.

The ordinal GCH representation achieved the highest Macro F1 (0.3303),
compared with 0.3272 for CAS-only, representing an improvement of 0.0030.
The categorical GCH representation produced a similar Macro F1 of 0.3299.

However, these improvements were accompanied by lower Balanced Accuracy
and lower Fatal-crash recall. The ordinal GCH configuration reduced Fatal
recall from 0.3231 to 0.2462.

Replacing the original CAS urban/open variable with categorical GCH23
produced a small Macro F1 improvement to 0.3290, but did not outperform
the CAS baseline across all evaluation metrics.

Therefore, although GCH23 showed a clear descriptive relationship with
crash severity, its incremental predictive contribution at seed 42 was
modest. Multi-seed evaluation is required before determining whether the
improvement is robust.

In [6]:
# GCH23 Five-Seed Robustness Experiment

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    recall_score
)

from xgboost import XGBClassifier


# 1. Seeds

seeds = [
    42,
    7,
    21,
    84,
    123
]


# 2. Configurations to test

robust_gch_sets = {

    "CAS Only":
        baseline_features,

    "CAS + GCH Ordinal":
        baseline_features
        + ["GCH_rurality_ordinal"],

    "CAS + GCH Categorical":
        baseline_features
        + ["GCH2023_clean"],

    "CAS + GCH Ordinal + Special Indicator":
        baseline_features
        + [
            "GCH_rurality_ordinal",
            "GCH_special_geography"
        ],

    "CAS Without Urban + GCH Categorical":
        baseline_without_urban
        + ["GCH2023_clean"]
}


# 3. Evaluation function

def evaluate_gch_seed(
    experiment_name,
    feature_list,
    seed
):

    indices = gch_model_df.index

    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=seed,
        stratify=gch_model_df.loc[
            indices,
            target
        ]
    )


    X_train = gch_model_df.loc[
        train_idx,
        feature_list
    ].copy()

    X_test = gch_model_df.loc[
        test_idx,
        feature_list
    ].copy()

    y_train = gch_model_df.loc[
        train_idx,
        target
    ].copy()

    y_test = gch_model_df.loc[
        test_idx,
        target
    ].copy()


    # Feature types

    categorical_list = [
        col
        for col in feature_list
        if (
            col in baseline_categorical
            or col == "GCH2023_clean"
        )
    ]

    numeric_list = [
        col
        for col in feature_list
        if col not in categorical_list
    ]


    # Preprocessing

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])


    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ])


    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            numeric_list
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_list
        )
    ])


    X_train_processed = (
        preprocessor.fit_transform(
            X_train
        )
    )

    X_test_processed = (
        preprocessor.transform(
            X_test
        )
    )


    # Target encoding

    encoder = LabelEncoder()

    y_train_encoded = (
        encoder.fit_transform(
            y_train
        )
    )

    y_test_encoded = (
        encoder.transform(
            y_test
        )
    )


    fatal_id = list(
        encoder.classes_
    ).index(
        "Fatal Crash"
    )

    serious_id = list(
        encoder.classes_
    ).index(
        "Serious Crash"
    )


    sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train_encoded
    )


    # Model

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=seed,
        n_jobs=-1
    )


    model.fit(
        X_train_processed,
        y_train_encoded,
        sample_weight=sample_weights
    )


    predictions = model.predict(
        X_test_processed
    )


    return {

        "Feature Set":
            experiment_name,

        "Seed":
            seed,

        "Accuracy":
            accuracy_score(
                y_test_encoded,
                predictions
            ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_test_encoded,
                predictions
            ),

        "Macro F1":
            f1_score(
                y_test_encoded,
                predictions,
                average="macro"
            ),

        "Fatal Recall":
            recall_score(
                y_test_encoded,
                predictions,
                labels=[fatal_id],
                average=None,
                zero_division=0
            )[0],

        "Serious Recall":
            recall_score(
                y_test_encoded,
                predictions,
                labels=[serious_id],
                average=None,
                zero_division=0
            )[0]
    }


# 4. Run all experiments

gch_robust_results = []

for seed in seeds:

    print(
        f"\nSeed {seed}"
    )

    for name, features in (
        robust_gch_sets.items()
    ):

        print(
            "Running:",
            name
        )

        gch_robust_results.append(
            evaluate_gch_seed(
                name,
                features,
                seed
            )
        )


gch_multiseed_raw = pd.DataFrame(
    gch_robust_results
)


print(
    "\nPer-Seed Results"
)

display(
    gch_multiseed_raw.round(4)
)


# 5. Summary

gch_multiseed_summary = (
    gch_multiseed_raw
    .groupby(
        "Feature Set"
    )
    .agg(
        Accuracy_Mean=(
            "Accuracy",
            "mean"
        ),

        Accuracy_SD=(
            "Accuracy",
            "std"
        ),

        Balanced_Accuracy_Mean=(
            "Balanced Accuracy",
            "mean"
        ),

        Balanced_Accuracy_SD=(
            "Balanced Accuracy",
            "std"
        ),

        Macro_F1_Mean=(
            "Macro F1",
            "mean"
        ),

        Macro_F1_SD=(
            "Macro F1",
            "std"
        ),

        Fatal_Recall_Mean=(
            "Fatal Recall",
            "mean"
        ),

        Fatal_Recall_SD=(
            "Fatal Recall",
            "std"
        ),

        Serious_Recall_Mean=(
            "Serious Recall",
            "mean"
        ),

        Serious_Recall_SD=(
            "Serious Recall",
            "std"
        )
    )
    .reset_index()
)


cas_macro = (
    gch_multiseed_summary.loc[
        gch_multiseed_summary[
            "Feature Set"
        ] == "CAS Only",
        "Macro_F1_Mean"
    ]
    .iloc[0]
)


gch_multiseed_summary[
    "Macro_F1_Change"
] = (
    gch_multiseed_summary[
        "Macro_F1_Mean"
    ]
    - cas_macro
)


# 6. Seeds improved vs corresponding CAS split

cas_seed_macro = (
    gch_multiseed_raw[
        gch_multiseed_raw[
            "Feature Set"
        ] == "CAS Only"
    ]
    .set_index(
        "Seed"
    )[
        "Macro F1"
    ]
)


seed_improvement = {}

for name in robust_gch_sets:

    if name == "CAS Only":

        seed_improvement[
            name
        ] = "-"

    else:

        experiment_macro = (
            gch_multiseed_raw[
                gch_multiseed_raw[
                    "Feature Set"
                ] == name
            ]
            .set_index(
                "Seed"
            )[
                "Macro F1"
            ]
        )

        improved = (
            experiment_macro
            > cas_seed_macro
        ).sum()

        seed_improvement[
            name
        ] = (
            f"{improved}/{len(seeds)}"
        )


gch_multiseed_summary[
    "Seeds Improved"
] = (
    gch_multiseed_summary[
        "Feature Set"
    ]
    .map(
        seed_improvement
    )
)


# 7. Sort results

gch_multiseed_summary = (
    gch_multiseed_summary
    .sort_values(
        "Macro_F1_Mean",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nFive-Seed Summary"
)

display(
    gch_multiseed_summary.round(4)
)


Seed 42
Running: CAS Only
Running: CAS + GCH Ordinal
Running: CAS + GCH Categorical
Running: CAS + GCH Ordinal + Special Indicator
Running: CAS Without Urban + GCH Categorical

Seed 7
Running: CAS Only
Running: CAS + GCH Ordinal
Running: CAS + GCH Categorical
Running: CAS + GCH Ordinal + Special Indicator
Running: CAS Without Urban + GCH Categorical

Seed 21
Running: CAS Only
Running: CAS + GCH Ordinal
Running: CAS + GCH Categorical
Running: CAS + GCH Ordinal + Special Indicator
Running: CAS Without Urban + GCH Categorical

Seed 84
Running: CAS Only
Running: CAS + GCH Ordinal
Running: CAS + GCH Categorical
Running: CAS + GCH Ordinal + Special Indicator
Running: CAS Without Urban + GCH Categorical

Seed 123
Running: CAS Only
Running: CAS + GCH Ordinal
Running: CAS + GCH Categorical
Running: CAS + GCH Ordinal + Special Indicator
Running: CAS Without Urban + GCH Categorical

Per-Seed Results


,Feature Set,Seed,Accuracy,Balanced Accuracy,Macro F1,Fatal Recall,Serious Recall
0,CAS Only,42,0.4900,0.4076,0.3272,0.3231,0.3892
1,CAS + GCH Ordinal,42,0.4984,0.3948,0.3303,0.2462,0.3892
2,CAS + GCH Categorical,42,0.4983,0.3922,0.3299,0.2308,0.3992
3,CAS + GCH Ordinal + Special Indicator,42,0.4947,0.3991,0.3296,0.2615,0.4052
4,CAS Without Urban + GCH Categorical,42,0.4953,0.3978,0.3290,0.2615,0.3972
5,CAS Only,7,0.4920,0.4111,0.3284,0.3385,0.3892
6,CAS + GCH Ordinal,7,0.4928,0.4028,0.3266,0.3077,0.3792
7,CAS + GCH Categorical,7,0.4917,0.4107,0.3279,0.3385,0.3852
8,CAS + GCH Ordinal + Special Indicator,7,0.4937,0.4047,0.3281,0.3077,0.3832
9,CAS Without Urban + GCH Categorical,7,0.4971,0.4144,0.3316,0.3385,0.3872



Five-Seed Summary


,Feature Set,Accuracy_Mean,Accuracy_SD,Balanced_Accuracy_Mean,Balanced_Accuracy_SD,Macro_F1_Mean,Macro_F1_SD,Fatal_Recall_Mean,Fatal_Recall_SD,Serious_Recall_Mean,Serious_Recall_SD,Macro_F1_Change,Seeds Improved
0,CAS Without Urban + GCH Categorical,0.5006,0.0052,0.4079,0.0061,0.3324,0.0030,0.2985,0.0354,0.3960,0.0160,0.0035,4/5
1,CAS + GCH Ordinal,0.4994,0.0044,0.4029,0.0059,0.3309,0.0037,0.2831,0.0319,0.3920,0.0203,0.0021,3/5
2,CAS + GCH Ordinal + Special Indicator,0.4964,0.0024,0.4075,0.0079,0.3307,0.0032,0.3015,0.0257,0.3984,0.0108,0.0019,3/5
3,CAS + GCH Categorical,0.4989,0.0049,0.4036,0.0098,0.3305,0.0039,0.2862,0.0443,0.3952,0.0170,0.0017,3/5
4,CAS Only,0.4940,0.0039,0.4062,0.0123,0.3288,0.0049,0.3077,0.0435,0.3944,0.0157,0.0000,-


### GCH23 Robustness and Feature-Representation Finding

Five-seed evaluation showed that the strongest use of GCH23 was not to add
it alongside the existing CAS urban/open variable, but to use the more
detailed GCH23 classification as a replacement for that variable.

The CAS-only model achieved a mean Macro F1 of 0.3288. Replacing the CAS
urban variable with categorical GCH23 increased mean Macro F1 to 0.3324
and improved Macro F1 in four of five random-seed experiments.

Mean Balanced Accuracy also increased slightly from 0.4062 to 0.4079, while
mean Serious-crash recall increased from 0.3944 to 0.3960. Mean Fatal-crash
recall decreased slightly from 0.3077 to 0.2985.

In contrast, adding GCH23 while retaining the original urban variable
produced smaller improvements. This indicates that GCH23 provides a richer
representation of geographic rurality context and may be more useful as a
replacement for the simpler binary CAS urban/open classification than as an
additional correlated feature.

The improvement remained modest, and the results do not indicate that GCH23
is superior across every evaluation metric.

In [7]:
# Save validated GCH23 crash-level enrichment

GCH_ENRICHED_PATH = (
    PROJECT_ROOT
    / "data"
    / "gch23"
    / "CAS_GCH23_enriched.csv"
)

gch_export = crash_gch[
    [
        "cas_row_id",
        "MB2023_V1_00",
        "crashSeverity",
        "urban",
        "GCH2023",
        "GCH2023_clean",
        "GCH_rurality_ordinal",
        "GCH_special_geography"
    ]
].copy()


gch_export.to_csv(
    GCH_ENRICHED_PATH,
    index=False
)


print(
    "Saved:"
)

print(
    GCH_ENRICHED_PATH
)


print(
    "\nShape:"
)

print(
    gch_export.shape
)


print(
    "\nDuplicate cas_row_id:"
)

print(
    gch_export[
        "cas_row_id"
    ]
    .duplicated()
    .sum()
)


print(
    "\nMissing GCH categorical:"
)

print(
    gch_export[
        "GCH2023_clean"
    ]
    .isna()
    .sum()
)


print(
    "\nMissing ordinal GCH:"
)

print(
    gch_export[
        "GCH_rurality_ordinal"
    ]
    .isna()
    .sum()
)

Saved:
C:\Users\ASUS\Desktop\Project_Roadcrash\data\gch23\CAS_GCH23_enriched.csv

Shape:
(34866, 8)

Duplicate cas_row_id:
0

Missing GCH categorical:
0

Missing ordinal GCH:
121
